Parte practica examen 1
Curso: Astrofísca Galáctica y Extragalactica

Fecha: 2026-10-04



Instalando las librerias necesarias

In [4]:
!pip install astropy numpy matplotlib scipy

### Ejercicio 1

Se provee una curva de rotación de la galaxia NGC2403. Además de esta curva, utilice un repositorio de imágenes astronómicas para obtener una imagen de la galaxia NGC2403 (mi recomendación es usar la banda g de SDSS).

Sólo puede usar la curva de rotación que se provee y las imágenes que consiga descargar desde los archivos.
Utilice un trazado de isofotas, ajuste de elipses a las mismas, etc. Con esos trazados de isofotas, diseñe métodos o use métodos ya conocidos para que:

- Defina cómo va a ubicar el centro de la galaxia.
- Estime la inclinación de la galaxia.
- Mida la distribución de brillo superficial de la galaxia (I(R)) y úsela para identificar la región donde inicia el bulbo y dónde termina el disco (zona de intersección).
- Haga una estimación a la clasificación de la galaxia en el sistema de Hubble.
- Trate de identificar los brazos espirales de la galaxia y proponga una manera de estimar sobre la imagen una medición del Pitch Angle (PA) de los brazos. Úsela y mida el PA de la galaxia.
- Use la información que se provee para estimar la distancia a la que se encuentra la galaxia.
Estime la escala radial del disco en Kpc.

Justifique el procedimiento por medio del cual midió/estimó cada uno de los ítems requeridos.

----

Importando las librerias

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from scipy.ndimage import gaussian_filter, label

Leyendo los datos

In [ ]:
# Leyendo los datos para ngc2403
import os

# En Jupyter, __file__ no está definido, pero podemos usar el directorio actual:
CARPETA = Path(os.getcwd())
RUTA_IMAGEN = CARPETA / "data" / "NGC_2403_beta_g.fits"
RUTA_CURVA = CARPETA / "data" / "NGC2403_Rot_Curve.dat"
CARPETA_FIGURAS = CARPETA / "figuras"

def cargar_curva(ruta):
    radio, velocidad = np.loadtxt(ruta, unpack=True)
    return radio, velocidad


def cargar_imagen(ruta):
    with fits.open(ruta) as hdul:
        imagen = hdul[0].data.astype(float)
        cabecera = hdul[0].header
    # CD1_1 esta en grados por pixel.
    escala_arco = abs(cabecera["CD1_1"]) * 3600.0
    return imagen, cabecera, escala_arco

def limpiar(imagen):
    """Resta el fondo y tapa estrellas puntuales con el valor suavizado."""
    
    # Quitando el 
    borde = np.concatenate(
        [
            imagen[:50, :].ravel(),
            imagen[-50:, :].ravel(),
            imagen[:, :50].ravel(),
            imagen[:, -50:].ravel(),
        ]
    )
    fondo = np.median(borde)
    
    sigma = 1.4826 * np.median(np.abs(borde - fondo))
    limpia = np.clip(imagen - fondo, 0, None)
    suave = gaussian_filter(limpia, 6)
    estrellas = (limpia - suave) > 8 * sigma
    limpia[estrellas] = suave[estrellas]
    return limpia, fondo, sigma

imagen, cabecera, escala_arco = cargar_imagen(RUTA_IMAGEN)
curva_r, curva_v = cargar_curva(RUTA_CURVA)
limpia, fondo, sigma = limpiar(imagen)

Variables

In [ ]:
# Espesor de un disco visto de canto. Sirve para pasar de b/a a inclinacion.
Q0 = 0.2
# Punto cero AB de un nanomaggy.
CERO_AB = 22.5
# Tully-Fisher en banda g (Pizagno et al. 2007):
#   M_g = pendiente * (log10(V) - log_v0) + cero
# V es la velocidad circular. Dispersion intrinseca ~ 0.46 mag.
TF_PENDIENTE = -5.476
TF_LOG_V0 = 2.220
TF_CERO = -20.686
TF_DISPERSION = 0.46
# Extincion interna aproximada: A_g = gamma * log10(a/b).
GAMMA_G = 1.5

Centro de la Galaxia:

In [ ]:
def momentos(mascara):
    """Elipse equivalente de una region: centro, semiejes y angulo."""
    ys, xs = np.nonzero(mascara)
    if xs.size < 200:
        return None
    xc = xs.mean()
    yc = ys.mean()
    dx = xs - xc
    dy = ys - yc
    mu20 = np.mean(dx * dx)
    mu02 = np.mean(dy * dy)
    mu11 = np.mean(dx * dy)
    valores, vectores = np.linalg.eigh([[mu20, mu11], [mu11, mu02]])
    orden = np.argsort(valores)[::-1]
    valores = valores[orden]
    vectores = vectores[:, orden]
    semimayor = 2 * np.sqrt(valores[0])
    semimenor = 2 * np.sqrt(valores[1])
    angulo = np.degrees(np.arctan2(vectores[1, 0], vectores[0, 0]))
    return xc, yc, semimayor, semimenor, semimenor / semimayor, angulo


def promedio_de_eje(angulos):
    """Promedio de un angulo de eje (da lo mismo sumar 180 grados)."""
    doble = np.deg2rad(2 * np.asarray(angulos, dtype=float))
    medio = np.arctan2(np.mean(np.sin(doble)), np.mean(np.cos(doble)))
    return np.degrees(medio) / 2

def isofotas(limpia, sigma):
    """
    Cada isofota es la region por encima de un nivel de brillo.
    La forma sale de los momentos de esa region.
    El centro es el centroide de las isofotas del disco, no el pixel mas brillante.
    """
    suave = gaussian_filter(limpia, 4)
    referencia = suave[suave > 5 * sigma]
    niveles = np.percentile(referencia, [15, 25, 35, 50, 65, 80])
    filas = []
    for nivel in niveles:
        regiones, n = label(limpia > nivel)
        if n == 0:
            continue
        conteo = np.bincount(regiones.ravel())
        conteo[0] = 0
        forma = momentos(regiones == np.argmax(conteo))
        if forma is None:
            continue
        filas.append(forma)

    # Disco: isofotas grandes y achatadas. El bulbo y el ruido quedan fuera.
    disco = [f for f in filas if 250 < f[2] < 1100 and 0.35 < f[4] < 0.75]
    if len(disco) < 2:
        disco = filas
    disco = sorted(disco, key=lambda f: f[2], reverse=True)
    externas = disco[:3]
    centro_x = np.median([f[0] for f in externas])
    centro_y = np.median([f[1] for f in externas])
    razon = np.median([f[4] for f in externas])
    angulo = promedio_de_eje([f[5] for f in externas])
    return filas, centro_x, centro_y, razon, angulo